# Sequnece Model Practice

- Conv1d + LSTM 기반 주가 예측 확장 실습:
 `Open`, `High`, `Low`, `Close`, `Volume`과 파생 feature를 함께 사용해 다변량 시계열 데이터를 구성한다.

## 1. 라이브러리 불러오기

필요한 라이브러리를 불러온다. `FinanceDataReader`가 설치되어 있지 않다면 아래 설치 코드를 먼저 실행한다.

In [ ]:
%pip install finance-datareader

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

import FinanceDataReader as fdr

## 2. 데이터 불러오기

In [ ]:
# 종목 코드 설정
naver_code = '035420'

# TODO: FinanceDataReader를 사용하여 네이버 주가 데이터를 가져오기
# naver = 

naver.head()

In [ ]:
# 데이터 기본 정보 확인
# TODO: 데이터의 행/열 개수, 결측치 개수, 최근 5개 행을 확인하기

## 3. feature 구성

다음 feature를 사용한다.

- `Open`: 시가
- `High`: 고가
- `Low`: 저가
- `Close`: 종가
- `Volume`: 거래량
- `MA5`: 5일 이동평균
- `MA20`: 20일 이동평균
- `Return`: 일별 수익률
- `Range`: 하루 변동폭 비율

정답값 `target`은 다음 날의 `Close`이다.

In [ ]:
# 원본 데이터 복사
stock_df = naver.copy()

# TODO: 5일 이동평균 MA5 생성
# stock_df['MA5'] = 

# TODO: 20일 이동평균 MA20 생성
# stock_df['MA20'] = 

# TODO: 일별 수익률 Return 생성
# stock_df['Return'] = 

# TODO: 하루 변동폭 비율 Range 생성
# stock_df['Range'] = 

# TODO: 결측치 제거
# stock_df = 

stock_df.head()

In [ ]:
# 사용할 feature 목록
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'MA5', 'MA20', 'Return', 'Range']
target_col = 'Close'

# TODO: feature 데이터와 target 데이터를 분리하기
# features = 
# target = 

print(features.shape)
print(target.shape)

## 4. 스케일링

feature와 target은 값의 범위가 다르다. 예를 들어 거래량은 가격보다 훨씬 큰 값을 가진다. 따라서 feature와 target을 각각 스케일링한다.


In [ ]:
# TODO: feature용 scaler와 target용 scaler를 각각 생성하기
# feature_scaler = 
# target_scaler = 

# TODO: feature와 target 각각 스케일링하기
# features_scaled = 
# target_scaled = 

print(features_scaled.shape)
print(target_scaled.shape)

## 5. 다변량 시계열 학습 데이터 생성

`window_size=20`이면 과거 20일의 여러 feature를 입력으로 사용하고, 그 다음 날의 종가를 정답으로 사용한다.

생성되는 입력 shape은 다음과 같다.

```text
X: (샘플 개수, window_size, feature 수)
y: (샘플 개수, 1)
```


In [ ]:
def create_multivariate_dataset(features, target, window_size=20):
    X, y = [], []

    # TODO: 과거 window_size일의 feature를 입력 X로 만들고,
    #       그 다음 날의 target을 y로 만들기
    
    return np.array(X), np.array(y)


window_size = 20
X, y = create_multivariate_dataset(features_scaled, target_scaled, window_size)

print(X.shape)
print(y.shape)

## 6. Tensor 변환 및 학습/검증 분리

시계열 데이터는 순서가 중요하므로 무작위 분리하지 않고 앞쪽 80%를 학습 데이터, 뒤쪽 20%를 검증 데이터로 사용한다.


In [ ]:
# TODO: X, y를 torch tensor로 변환하기
# X_tensor = 
# y_tensor = 

# TODO: 앞쪽 80%를 학습 데이터로 사용하기
# train_size = 
# X_train, X_val = 
# y_train, y_val = 

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)

## 7. Conv1d + LSTM 모델 정의

`input_size`는 feature 개수이다. 
즉 `Open`, `High`, `Low`, `Close`, `Volume`, `MA5`, `MA20`, `Return`, `Range`를 사용하면 `input_size=9`가 된다.

Conv1d는 `(batch_size, channel, seq_len)` 형태를 요구한다. 하지만 LSTM용 데이터는 `(batch_size, seq_len, feature 수)` 형태이므로 `permute()`가 필요하다.

In [ ]:
class StockConvLSTM(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, conv_filters, conv_kernel_size):
        super().__init__()

        # TODO: Conv1d 레이어 정의하기
        # in_channels는 입력 feature 수이다.
        # out_channels는 Conv1d가 만들어낼 특징 수이다.
        # self.conv1d = 

        # TODO: LSTM 레이어 정의하기
        # LSTM의 input_size는 conv_filters가 된다.
        # self.lstm = 

        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        # x: (batch_size, seq_len, input_size)

        # TODO: Conv1d 입력 형태로 변환하기
        # x = 

        # TODO: Conv1d 적용하기
        # x = 

        # TODO: LSTM 입력 형태로 다시 변환하기
        # x = 

        # TODO: LSTM 적용 후 마지막 hidden state 사용하기
        # _, (hidden, _) = 

        output = self.fc1(hidden[-1])
        output = self.relu(output)
        output = self.fc2(output)
        return output

## 8. 모델 생성 및 학습

`input_size`는 feature 개수인 `len(feature_cols)`로 설정한다.


In [ ]:
# TODO: 모델 생성하기
# input_size는 len(feature_cols)이다.
# model = 

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 50
batch_size = 32
train_losses = []
val_losses = []

for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(X_train.size(0))
    train_loss = 0

    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i + batch_size]
        batch_X = X_train[indices]
        batch_y = y_train[indices]

        # TODO: 예측, 손실 계산, 역전파, 파라미터 업데이트 작성하기
        
    avg_train_loss = train_loss / X_train.size(0)
    train_losses.append(avg_train_loss)

    model.eval()
    with torch.no_grad():
        # TODO: 검증 데이터 손실 계산하기
        # val_pred = 
        # val_loss = 
        pass

    val_losses.append(val_loss.item())

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Train Loss: {avg_train_loss:.6f}, Val Loss: {val_loss.item():.6f}')

## 9. 예측 결과 복원 및 시각화

모델 출력은 스케일링된 값이다. 실제 주가 단위로 해석하려면 `target_scaler.inverse_transform()`을 사용해 복원해야 한다.


In [ ]:
model.eval()
with torch.no_grad():
    # TODO: 검증 데이터 예측하기
    # pred_scaled = 
    pass

# TODO: 예측값과 실제값을 원래 주가 단위로 복원하기
# pred_inverse = 
# y_val_inverse = 

print(pred_inverse[:5].flatten())
print(y_val_inverse[:5].flatten())

In [ ]:
# TODO: 실제값과 예측값을 선 그래프로 비교하기

In [ ]:
# TODO: MAE, RMSE 계산하기
# mae = 
# rmse = 

print('MAE:', mae)
print('RMSE:', rmse)

## 10. 다음 날 종가 예측 함수 만들기

최근 `window_size`일의 feature 데이터를 입력받아 다음 날 종가를 예측하는 함수를 만든다.


In [ ]:
def predict_next_close(stock_df, model, feature_cols, feature_scaler, target_scaler, window_size=20):
    # 최근 window_size일의 feature만 사용
    # TODO: feature 데이터 추출
    # recent_features = 

    # TODO: feature 스케일링
    # recent_scaled = 

    # TODO: tensor 변환
    # input_tensor = 

    model.eval()
    with torch.no_grad():
        # TODO: 모델 예측
        # pred_scaled = 
        pass

    # TODO: 예측값 역스케일링
    # pred_inverse = 

    return pred_inverse[0, 0]


# TODO: 다음 날 종가 예측 실행하기
# next_price = 
# print(next_price)